# Generate a noisy lower-power TIFF

This notebook retains the synthetic acquisition that the Biowulf SNR workflow normally creates only as a temporary file. It runs locally on macOS and does not need Slurm, a GPU, or a SUPPORT training run.

The calculation matches `scripts/biowulf_snr_test.sbatch` / `notebooks/snr_test.py`:

1. The raw and original SUPPORT-denoised TIFFs are used together to estimate the effective detected-count scale.
2. The **denoised** TIFF is treated as the underlying signal. Measured excitation power is converted to expected two-photon signal with `signal_ratio = (power_percent / 100) ** 2`.
3. Independent Poisson samples create a synthetic raw-looking, lower-SNR TIFF. The original raw pixels are calibration data; noise is not added to those already-noisy pixels. Even 100% power creates a new Poisson sample.

Edit the four values in the next cell, then run all cells in order. The default strict calibration trims 60 frames from each end and needs at least 140 time points. A 124–139-frame movie has enough frames for the calculation but triggers the production workflow's too-few-pairs warning and stops unless provisional estimates are explicitly allowed.

In [ ]:
from pathlib import Path

# -------------------------------------------------------------------------
# EDIT THESE FOUR VALUES
# -------------------------------------------------------------------------
RAW_TIFF = Path('/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260921_snrTest/01_data/01_raw/6xFAD_TRD_GC8_M205_20251119_FOV1_4D/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2.tif')
DENOISED_TIFF = Path('/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260921_snrTest/01_data/03_denoised/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2_original/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2_original_SPRT.tif')
OUTPUT_DIRECTORY = Path('/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260921_snrTest/01_data/02_lowerPowerRaw/6xFAD_TRD_GC8_M205_20251119_FOV1_4D_Ch2')
POWER_PERCENT = 50

# The settings below match notebooks/snr_test.py. Usually leave them alone.
FRAME_START = 60
FRAME_END_MARGIN = 60
MAX_PAIRS = 50
SAMPLE_COUNT = 40_000
CALIBRATION_SEED = 7
SIMULATION_SEED = 1
ZERO_PHOTON_LEVEL = 0.0
USE_ESTIMATED_PHOTON_PEDESTAL = False
ALLOW_PROVISIONAL_PHOTON_ESTIMATE = False
OVERWRITE_EXISTING = False
TIME_CHUNK_SIZE = 1

# Set this only if Jupyter was started outside the SUPPORT repository tree.
# Example: Path('/Users/your_name/Downloads/SUPPORT')
SUPPORT_REPOSITORY = None

## Advanced settings

`ZERO_PHOTON_LEVEL` is a fixed detector/digitizer baseline in SUPPORT-image units, not an optical-background estimate. Keep it at zero unless a dark or laser-off acquisition provides a better value. `USE_ESTIMATED_PHOTON_PEDESTAL=True` is a model-dependent fallback and cannot be combined with a nonzero fixed value.

The production workflow stops when its calibration diagnostics emit warnings. This notebook does the same. Set `ALLOW_PROVISIONAL_PHOTON_ESTIMATE=True` only after reading the printed warnings and deciding that a visual, provisional result is useful.

`TIME_CHUNK_SIZE=1` applies the same `simulate_lower_power` function a time point at a time with one persistent random generator. The result is written through a disk-backed TIFF, which avoids holding the whole synthetic movie and the Poisson intermediates in Mac memory. Increasing it can improve speed when ample RAM is available.

In [ ]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile


def find_support_repository(start):
    """Find the repository whether Jupyter starts in SUPPORT/ or tests/."""
    start = Path(start).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'utils' / 'salt.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Could not find the SUPPORT repository above the Jupyter working '
        f'directory ({start}). Start Jupyter from the SUPPORT repository or '
        'its tests directory.'
    )


if SUPPORT_REPOSITORY is None:
    REPOSITORY = find_support_repository(Path.cwd())
else:
    REPOSITORY = Path(SUPPORT_REPOSITORY).expanduser().resolve()
    if not (REPOSITORY / 'utils' / 'salt.py').is_file():
        raise FileNotFoundError(
            f'SUPPORT_REPOSITORY does not contain utils/salt.py: {REPOSITORY}'
        )
if str(REPOSITORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY))

from utils.salt import estimate_photons_per_unit, simulate_lower_power

print(f'Using SUPPORT repository: {REPOSITORY}')

In [ ]:
def resolve_input_tiff(path, label):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'{label} does not exist or is not a file: {path}')
    if path.suffix.casefold() not in {'.tif', '.tiff'}:
        raise ValueError(f'{label} must be a .tif or .tiff file: {path}')
    return path


def read_time_first_tiff(path):
    """Read TIFF series 0 and put a metadata-declared T axis first."""
    path = Path(path).resolve()
    with tifffile.TiffFile(path) as tif:
        axes = tif.series[0].axes

    try:
        array = tifffile.memmap(path, series=0, mode='r')
    except (OSError, ValueError):
        # Compressed/tiled TIFFs cannot be mapped in place. tifffile decodes
        # these to a temporary memory map instead of filling ordinary RAM.
        array = tifffile.imread(path, series=0, out='memmap')

    normalized_declared_axes = None
    if axes and 'T' in axes:
        time_axis = axes.index('T')
        if time_axis != 0:
            array = np.moveaxis(array, time_axis, 0)
        normalized_declared_axes = 'T' + axes[:time_axis] + axes[time_axis + 1:]
    elif array.ndim in (3, 4):
        warnings.warn(
            f'{path.name} has axes={axes!r}; assuming axis 0 is time.',
            stacklevel=2,
        )
    else:
        raise ValueError(
            f'Cannot identify a time axis for {path}: '
            f'axes={axes!r}, shape={array.shape}'
        )

    if array.ndim not in (3, 4):
        raise ValueError(
            f'{path.name} must be time-first TYX or TZYX after axis '
            f'normalization; got shape {array.shape}.'
        )
    canonical_axes = 'TZYX' if array.ndim == 4 else 'TYX'
    if (
        normalized_declared_axes is not None
        and normalized_declared_axes != canonical_axes
    ):
        raise ValueError(
            f'{path.name} declares axes={axes!r}, which normalize to '
            f'{normalized_declared_axes!r}. This workflow supports only '
            'TYX and TZYX movies.'
        )
    return array, canonical_axes


def simulate_to_tiff_atomically(
    path,
    denoised_stack,
    axes,
    *,
    power_ratio,
    photons_per_unit,
    zero_photon_level,
    seed,
    time_chunk_size,
    overwrite,
):
    """Run the production simulator in bounded-memory time chunks."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f'.{path.name}.', suffix='.tmp.tif', dir=path.parent
    )
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    try:
        output_stack = tifffile.memmap(
            temporary_path,
            shape=denoised_stack.shape,
            dtype=np.float32,
            imagej=True,
            metadata={'axes': axes},
        )
        random_generator = np.random.default_rng(seed)
        time_count = denoised_stack.shape[0]
        chunk_count = (time_count + time_chunk_size - 1) // time_chunk_size
        progress_interval = max(1, chunk_count // 10)
        for chunk_index, frame_start in enumerate(
            range(0, time_count, time_chunk_size)
        ):
            frame_stop = min(frame_start + time_chunk_size, time_count)
            # default_rng returns a supplied Generator unchanged. Passing one
            # persistent Generator therefore produces the same random stream
            # while the public production helper operates on one chunk.
            output_stack[frame_start:frame_stop] = simulate_lower_power(
                denoised_stack[frame_start:frame_stop],
                power_ratio=power_ratio,
                gamma=2.0,
                photons_per_unit=photons_per_unit,
                seed=random_generator,
                zero_photon_level=zero_photon_level,
            )
            completed_chunks = chunk_index + 1
            if (
                completed_chunks % progress_interval == 0
                or completed_chunks == chunk_count
            ):
                print(
                    f'  simulated frames {frame_stop:,}/{time_count:,} '
                    f'({completed_chunks:,}/{chunk_count:,} chunks)'
                )
        output_stack.flush()
        del output_stack

        if overwrite:
            os.replace(temporary_path, path)
        else:
            try:
                # A hard link publishes atomically without overwriting a file
                # that appeared after the earlier existence check.
                os.link(temporary_path, path)
            except FileExistsError as exc:
                raise FileExistsError(
                    f'Output appeared while the simulation was running: {path}'
                ) from exc
            except OSError:
                # exFAT and some SMB/NAS mounts do not support hard links.
                # Atomically reserve the destination name, then replace only
                # the empty file created by this process.
                try:
                    reservation_descriptor = os.open(
                        path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600
                    )
                except FileExistsError as exc:
                    raise FileExistsError(
                        f'Output appeared while the simulation was running: {path}'
                    ) from exc
                reservation_stat = os.fstat(reservation_descriptor)
                os.close(reservation_descriptor)
                reservation_identity = (
                    reservation_stat.st_dev,
                    reservation_stat.st_ino,
                )
                try:
                    os.replace(temporary_path, path)
                except BaseException:
                    try:
                        current_stat = os.stat(path, follow_symlinks=False)
                    except FileNotFoundError:
                        pass
                    else:
                        current_identity = (current_stat.st_dev, current_stat.st_ino)
                        if current_identity == reservation_identity:
                            path.unlink()
                    raise
            else:
                temporary_path.unlink()
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    return tifffile.memmap(path, series=0, mode='r')


## Calibrate, simulate, and save

The output is named `<raw stem>_original_<power>.tif`, matching the temporary simulated-acquisition name used by the SNR sweep, and is written directly inside `OUTPUT_DIRECTORY`.

In [ ]:
raw_path = resolve_input_tiff(RAW_TIFF, 'RAW_TIFF')
denoised_path = resolve_input_tiff(DENOISED_TIFF, 'DENOISED_TIFF')
output_directory = Path(OUTPUT_DIRECTORY).expanduser().resolve()
if raw_path == denoised_path:
    raise ValueError('RAW_TIFF and DENOISED_TIFF must be different files.')

if isinstance(POWER_PERCENT, (bool, np.bool_)) or not isinstance(
    POWER_PERCENT, (int, np.integer)
):
    raise TypeError('POWER_PERCENT must be an integer from 1 through 100.')
power_percent = int(POWER_PERCENT)
if not 1 <= power_percent <= 100:
    raise ValueError('POWER_PERCENT must be from 1 through 100.')
if output_directory.exists() and not output_directory.is_dir():
    raise NotADirectoryError(
        f'OUTPUT_DIRECTORY exists but is not a directory: {output_directory}'
    )
output_path = output_directory / (
    f'{raw_path.stem}_original_{power_percent:03d}.tif'
)
if output_path in {raw_path, denoised_path}:
    raise ValueError(f'Refusing to replace an input TIFF: {output_path}')
if output_path.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(
        f'Output already exists: {output_path}. Set OVERWRITE_EXISTING=True '
        'to replace it intentionally.'
    )
if FRAME_START < 0 or FRAME_END_MARGIN < 0:
    raise ValueError('FRAME_START and FRAME_END_MARGIN must be nonnegative.')
if MAX_PAIRS <= 0 or SAMPLE_COUNT <= 0:
    raise ValueError('MAX_PAIRS and SAMPLE_COUNT must be positive.')
if isinstance(TIME_CHUNK_SIZE, (bool, np.bool_)) or not isinstance(
    TIME_CHUNK_SIZE, (int, np.integer)
) or TIME_CHUNK_SIZE <= 0:
    raise ValueError('TIME_CHUNK_SIZE must be a positive integer.')
if not np.isfinite(ZERO_PHOTON_LEVEL):
    raise ValueError('ZERO_PHOTON_LEVEL must be finite.')
if USE_ESTIMATED_PHOTON_PEDESTAL and ZERO_PHOTON_LEVEL != 0.0:
    raise ValueError(
        'Choose either USE_ESTIMATED_PHOTON_PEDESTAL=True or a nonzero '
        'ZERO_PHOTON_LEVEL, not both.'
    )

raw_stack, raw_axes = read_time_first_tiff(raw_path)
denoised_stack, denoised_axes = read_time_first_tiff(denoised_path)
if raw_stack.shape != denoised_stack.shape:
    raise ValueError(
        'The raw and denoised TIFFs must have the same time-first shape; '
        f'got {raw_stack.shape} and {denoised_stack.shape}.'
    )
if raw_axes != denoised_axes:
    raise ValueError(f'Axis mismatch: raw={raw_axes}, denoised={denoised_axes}.')

chunk_frames = min(int(TIME_CHUNK_SIZE), denoised_stack.shape[0])
float32_bytes = np.dtype(np.float32).itemsize
output_gib = denoised_stack.size * float32_bytes / 1024 ** 3
chunk_gib = (
    chunk_frames * np.prod(denoised_stack.shape[1:]) * float32_bytes
    / 1024 ** 3
)

frame_stop = raw_stack.shape[0] - int(FRAME_END_MARGIN)
if frame_stop - int(FRAME_START) < 4:
    raise ValueError(
        'The configured calibration window must contain at least four '
        f'frames; got [{FRAME_START}, {frame_stop}) from '
        f'{raw_stack.shape[0]} total frames.'
    )

print(f'Raw TIFF:      {raw_path}')
print(f'Denoised TIFF: {denoised_path}')
print(f'Shape/axes:    {raw_stack.shape} / {raw_axes}')
print(f'Output size:   about {output_gib:.2f} GiB on disk')
print(
    f'Simulation:  {chunk_frames} time point(s) per chunk; the largest '
    f'float32 chunk is {chunk_gib:.2f} GiB before temporary arrays'
)
print(
    f'Calibrating on frames [{FRAME_START}, {frame_stop}) with up to '
    f'{MAX_PAIRS} adjacent pairs ...'
)
estimate = estimate_photons_per_unit(
    raw_stack,
    denoised_stack,
    frame_start=int(FRAME_START),
    frame_stop=frame_stop,
    max_pairs=int(MAX_PAIRS),
    sample_count=int(SAMPLE_COUNT),
    random_seed=int(CALIBRATION_SEED),
)

print()
print('Photon calibration:')
print(f'  photons_per_unit           = {estimate.photons_per_unit:.9g}')
print(f'  photons_per_raw_unit       = {estimate.photons_per_raw_unit:.9g}')
print(f'  SUPPORT gain               = {estimate.support_gain:.7g}')
print(f'  effective SUPPORT pedestal = {estimate.effective_support_pedestal:.7g}')
print(f'  variance R^2               = {estimate.linear_r_squared:.6f}')
print(f'  variance delta AICc        = {estimate.delta_aicc:+.3f}')
print(f'  scale R^2                  = {estimate.scale_r_squared:.6f}')
print(f'  scale delta AICc           = {estimate.scale_delta_aicc:+.3f}')
if estimate.warnings:
    print('  warnings:')
    for warning in estimate.warnings:
        print(f'    - {warning}')
    if not ALLOW_PROVISIONAL_PHOTON_ESTIMATE:
        raise RuntimeError(
            'Photon calibration failed the production quality checks. '
            'Review the warnings above. To create a visual-only result '
            'anyway, set ALLOW_PROVISIONAL_PHOTON_ESTIMATE=True and rerun.'
        )

zero_photon_level = (
    float(estimate.effective_support_pedestal)
    if USE_ESTIMATED_PHOTON_PEDESTAL
    else float(ZERO_PHOTON_LEVEL)
)
power_ratio = power_percent / 100.0
print()
print(
    f'Simulating {power_percent}% excitation power '
    f'({power_ratio ** 2:.2%} expected photon-generating intensity) ...'
)
simulated_stack = simulate_to_tiff_atomically(
    output_path,
    denoised_stack,
    denoised_axes,
    power_ratio=power_ratio,
    photons_per_unit=float(estimate.photons_per_unit),
    zero_photon_level=zero_photon_level,
    seed=int(SIMULATION_SEED),
    time_chunk_size=int(TIME_CHUNK_SIZE),
    overwrite=bool(OVERWRITE_EXISTING),
)

with tifffile.TiffFile(output_path) as saved_tiff:
    saved_series = saved_tiff.series[0]
    saved_shape = saved_series.shape
    saved_dtype = saved_series.dtype
    saved_axes = saved_series.axes
if (
    saved_shape != simulated_stack.shape
    or saved_dtype != np.dtype(np.float32)
    or saved_axes != denoised_axes
):
    raise RuntimeError(
        f'Saved TIFF validation failed: shape={saved_shape}, '
        f'dtype={saved_dtype}, axes={saved_axes!r}.'
    )
print()
print(f'Saved synthetic noisy TIFF: {output_path}')
print(f'Saved shape/dtype/axes: {saved_shape} / {saved_dtype} / {saved_axes}')

## Preview a representative plane

For a TZYX movie, this displays the middle Z plane of the middle time point. Each panel uses its own robust contrast range because raw and SUPPORT-image intensity units may differ. The full result is the TIFF printed above.

In [ ]:
from matplotlib.pyplot import rcParams
rcParams['figure.dpi'] = 300
rcParams['font.family'] = 'SF Pro Rounded'

In [ ]:
def representative_plane(stack):
    plane = np.asarray(stack[stack.shape[0] // 2])
    if plane.ndim == 3:
        plane = plane[plane.shape[0] // 2]
    return plane


preview_items = (
    ('Raw calibration input', representative_plane(raw_stack)),
    ('Original denoised', representative_plane(denoised_stack)),
    (f'Synthetic {power_percent}% power', representative_plane(simulated_stack)),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for axis, (title, plane) in zip(axes, preview_items):
    finite = plane[np.isfinite(plane)]
    if finite.size:
        vmin, vmax = np.percentile(finite, (1, 99))
        if vmax <= vmin:
            vmin, vmax = float(finite.min()), float(finite.max())
    else:
        vmin, vmax = None, None
    image = axis.imshow(plane, cmap='gray', vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
plt.show()